In [7]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import os
import numpy as np


In [4]:
import pickle

In [8]:
data_path = "../../data/hcm_data/"
train_path = os.path.join(data_path, "train.npy")
data = np.load(train_path, allow_pickle=True)
print(f"Loaded data from {train_path}, shape: {data.shape}")

Loaded data from ../../data/hcm_data/train.npy, shape: (765036, 6)


In [9]:
with open(os.path.join(data_path,"nwk_hcm/hcm_edges_poi_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)

with open(os.path.join(data_path,"nwk_hcm/hcm_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [10]:
sample_trip = data[0]
print(f"Sample trip data: {sample_trip}")

Sample trip data: [19497
 list([21264, 4999, 21263, 26818, 5309, 43114, 25057, 5690, 43455, 21314, 25072, 25074, 21302, 25133, 21299, 12074, 42247, 42248, 42250, 42252, 5449, 18145, 21309, 44724, 25823, 1194, 21863, 21857, 21368, 21257, 22187, 45642, 22166, 26287, 39457, 26277, 3565, 4710, 14057, 25818, 4433, 21258, 44439, 11330, 11329, 3909, 10039, 10037, 9454, 32938, 10040, 17554, 532, 46138, 40651, 45702, 12573, 45257, 12577, 45247, 12579, 45238, 45233, 12571, 12563, 12568, 40410, 12566, 17982, 45216, 45250, 2074, 9655, 45262, 9649, 10118, 10068, 40374, 26662, 4562, 26207, 13916, 13915, 17508, 40379, 27623, 22521, 22517, 23062, 16843, 23057, 23063, 23068, 23069, 23074, 22512, 22514, 22531, 16969, 22530, 22536, 22535, 28001, 27374, 12220, 4058, 16802, 11411, 11410, 31626, 12222, 11406, 16793, 30095, 32413, 29987, 30102, 44166, 32415, 20886, 23573, 871, 23570, 29255, 14204, 29243, 2543, 23095, 4690, 28536, 18905, 23108, 37830, 37834, 28533, 20911, 28465, 28476, 16674, 37776, 1504, 189

In [11]:
for edge in sample_trip[1]:
    print(f"Edge: {edge}")
    if edge in edgeinfo:
        print(f"Edge info: {edgeinfo[edge]}")
    else:
        print("Edge not found in edgeinfo.")

Edge: 21264
Edge info: ['primary', 81.87268659283262, '4875861250', '411918284', 6.0, 0.0, 2.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 2.0]
Edge: 4999
Edge info: ['primary', 81.87268659283262, '411918284', '4875861250', 6.0, 0.0, 2.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 2.0]
Edge: 21263
Edge info: ['primary', 21.750539944538332, '4875861250', '5518776427', 12.0, 0.0, 6.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 4.0]
Edge: 26818
Edge info: ['primary', 198.81163162813206, '5518776427', '411919410', 12.0, 0.0, 6.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 4.0]
Edge: 5309
Edge info: ['primary', 65.9773672857416, '411919410', '8366432224', 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0]
Edge: 43114
Edge info: ['primary', 25.80613211348403, '8366432224', '5074798925', 18.0, 0.0, 6.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.0]
Edge: 25057
Edge info: ['primary', 39.98817217761779, '5074798925', '411923498', 18.0, 0.0, 6.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.0]
Edge: 5690
Edge info: ['primary', 20.060735080950764, '411923498', '8771118197

In [6]:
print(edgeinfo[1])

['residential', 177.32751993494392, '366367223', '366413600', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
import networkx as nx  # or compute manually

# edgeinfo: dict of {idx: [highway, length, node_u, node_v, poi*9, ...]}

# Build degree maps by scanning all edges
in_degree = {}   # node -> count of edges where node_v == node
out_degree = {}  # node -> count of edges where node_u == node

for idx, edge in edgeinfo.items():
    node_u = edge[2]
    node_v = edge[3]
    out_degree[node_u] = out_degree.get(node_u, 0) + 1
    in_degree[node_v]  = in_degree.get(node_v, 0) + 1

# Inject into edgeinfo
for idx, edge in edgeinfo.items():
    node_u = edge[2]
    node_v = edge[3]
    deg_out = out_degree.get(node_u, 0)  # how many edges leave node_u
    deg_in  = in_degree.get(node_v, 0)   # how many edges arrive at node_v
    edge.append(deg_out)
    edge.append(deg_in)

In [ ]:
import numpy as np

deg_in_vals  = [edge[13] for edge in edgeinfo.values()]
deg_out_vals = [edge[14] for edge in edgeinfo.values()]

for name, vals in [("deg_in", deg_in_vals), ("deg_out", deg_out_vals)]:
    arr = np.array(vals)
    print(f"{name}: min={arr.min()}, max={arr.max()}, mean={arr.mean():.2f}, "
          f"median={np.median(arr):.1f}, std={arr.std():.2f}, "
          f"p95={np.percentile(arr,95):.1f}, p99={np.percentile(arr,99):.1f}")

In [ ]:
sample_edge = edgeinfo[2]
sample_u = sample_edge[2]
sample_v = sample_edge[3]

In [ ]:
unique_edges = set()

for edge in edgeinfo.values():
    u = edge[2]
    v = edge[3]
    unique_edges.add((min(u, v), max(u, v)))

In [ ]:
total_edges = len(edgeinfo)
unique_undirected = len(unique_edges)

print("total_edges:", total_edges)
print("unique_undirected:", unique_undirected)
print("ratio:", total_edges / unique_undirected)

In [ ]:
deg = {}

for edge in unique_edges:
    u = edge[0]
    v = edge[1]

    deg[u] = deg.get(u, 0) + 1
    deg[v] = deg.get(v, 0) + 1

In [ ]:
edge_deg_start = []
edge_deg_end = []

for edge in edgeinfo.values():
    u = edge[2]
    v = edge[3]

    edge_deg_start.append(deg[u])
    edge_deg_end.append(deg[v])

In [ ]:
import numpy as np

deg_in_vals = np.array(edge_deg_start)
deg_out_vals = np.array(edge_deg_end)

def print_stats(name, arr):
    print(f"\n{name} stats:")
    print(f"  count: {len(arr)}")
    print(f"  mean: {arr.mean():.4f}")
    print(f"  std: {arr.std():.4f}")
    print(f"  min: {arr.min()}")
    print(f"  max: {arr.max()}")
    print(f"  median: {np.median(arr)}")

print_stats("deg_in", deg_in_vals)
print_stats("deg_out", deg_out_vals)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(deg_in_vals, bins=50)
plt.title("In-degree distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.hist(deg_out_vals, bins=50)
plt.title("Out-degree distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure()
plt.hist(deg_in_vals, bins=50)
plt.yscale("log")
plt.title("In-degree distribution (log scale)")
plt.show()